In [ ]:
pip install xgboost

In [ ]:
import numpy as np 
import pandas as pd 
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Load the dataset
df = pd.read_csv('/kaggle/input/airline-dataset/Airline Dataset.csv')  # Replace 'your_dataset.csv' with the actual path to your file

# Data Cleaning: Remove null values if any
df.dropna(inplace=True)

# Feature Engineering: Extract day of the week, month from Departure Date
df['Departure Date'] = pd.to_datetime(df['Departure Date'])
df['Day_of_Week'] = df['Departure Date'].dt.dayofweek
df['Month'] = df['Departure Date'].dt.month

# Prepare the features
features = ['Airport Name', 'Airport Country Code', 'Country Name', 'Airport Continent', 'Day_of_Week', 'Month', 'Pilot Name']
X = df[features].copy()  # Create a copy to avoid SettingWithCopyWarning

# Label Encoding for categorical features
label_encoders = {}
for feature in ['Airport Name', 'Airport Country Code', 'Country Name', 'Airport Continent', 'Pilot Name']:
    le = LabelEncoder()
    X.loc[:, feature] = le.fit_transform(X[feature])  # Use .loc to avoid SettingWithCopyWarning
    label_encoders[feature] = le

# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Prepare the target variable
y = df['Flight Status'].apply(lambda x: 1 if x == 'Delayed' else 0)  # Adjust this based on your actual data

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42)

# Initialize and train the XGBoost classifier
xgb = XGBClassifier(eval_metric='logloss')
xgb.fit(X_train, y_train)

# Make predictions
y_pred = xgb.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Model Accuracy: {accuracy}")
print(classification_report(y_test, y_pred))
